# SAE Training with W&B

Train the sparse autoencoder on SLformer embeddings, stream epoch metrics to Weights & Biases, and save the local checkpoints used by downstream SAE analysis.


In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import wandb
from sklearn.model_selection import train_test_split

# Reference-style notebook hygiene
warnings.filterwarnings("ignore", message="IProgress not found*")
warnings.filterwarnings("ignore", message=r"The epoch parameter in `scheduler\.step\(\)` was not necessary*")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "SAE").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = (REPO_ROOT / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from SAE.SAE_training.model import SAEConfig, SparseAutoencoder
from SAE.SAE_training.utils.data import build_artifacts, extract_concat_matrix
from SAE.SAE_training.utils.diagnostics import corr_offdiag, effective_rank, summ
from SAE.SAE_training.utils.plotting import load_metrics, plot_training_summary
from SAE.SAE_training.utils.training import fit_and_save_sae, load_sae_train_config, setup_experiment

sns.set_theme(style="whitegrid")
print("Repo root:", REPO_ROOT)
print("Using src dir:", SRC_DIR)


In [ ]:
# --- Load config, set run directory, and initialize W&B ---
CONFIG_PATH = (REPO_ROOT / "src" / "SAE" / "SAE_training" / "config" / "train_config.yaml").resolve()
cfg, model_cfg, train_cfg = load_sae_train_config(CONFIG_PATH)

paths_cfg = cfg["paths"]
scope_cfg = cfg["scope"]
norm_cfg = cfg["normalization"]
wandb_cfg = cfg["wandb"]

embeddings_pkl = Path(paths_cfg["embeddings_pkl"]).expanduser().resolve()
prediction_csvs = [Path(p).expanduser().resolve() for p in paths_cfg["prediction_csvs"]]
output_base_dir = Path(paths_cfg["output_base_dir"]).expanduser().resolve()
slformer_output_subdir = str(paths_cfg["slformer_output_subdir"])

CANCER = str(scope_cfg["cancer"])
MAX_SAMPLES = scope_cfg["max_samples"]
SCORE_COL = str(scope_cfg["score_col"])
SEED = int(scope_cfg["seed"])
VAL_SPLIT = float(cfg["train"]["val_split"])
EPS = float(norm_cfg["eps"])

setup_experiment(seed=train_cfg.seed)

run_name = f"hidden{model_cfg.d_hidden}_gate{model_cfg.gate_weight}_orth{model_cfg.orth_weight}_k{model_cfg.topk}"
run_dir = output_base_dir / slformer_output_subdir / CANCER / run_name
run_dir.mkdir(parents=True, exist_ok=True)

os.environ["WANDB_API_KEY"] = wandb_cfg["key"]
wandb_run = wandb.init(
    project=str(wandb_cfg["project"]),
    entity=wandb_cfg["entity"],
    mode=str(wandb_cfg["mode"]),
    name=f"{CANCER}_{run_name}",
    group=CANCER,
    tags=list(wandb_cfg["tags"]),
    dir=str(run_dir),
    config={
        "config_path": str(CONFIG_PATH),
        "scope": scope_cfg,
        "normalization": norm_cfg,
        "model": asdict(model_cfg),
        "train": asdict(train_cfg),
    },
)

print("Config:", CONFIG_PATH)
print("Embeddings:", embeddings_pkl)
print("Pred CSVs:", prediction_csvs[0], "...", prediction_csvs[-1])
print("Output run dir:", run_dir)
print("W&B run:", None if wandb_run is None else wandb_run.url)
print("Cancer:", CANCER, "Val split:", VAL_SPLIT)
print("AMP:", train_cfg.amp, "Scheduler:", train_cfg.scheduler_type)
print("Gate weight:", model_cfg.gate_weight, "TopK:", model_cfg.topk, "Jump threshold:", model_cfg.jump_threshold)
print("CKPT:", train_cfg.save_ckpt, "every", train_cfg.ckpt_every_n_epochs, "Resume:", train_cfg.use_saved_ckpt)


In [ ]:
# --- Load selected dataset (concat to 1024-d) ---
artifacts = build_artifacts(embeddings_pkl=embeddings_pkl, prediction_csvs=prediction_csvs)
X, y, meta = extract_concat_matrix(
    artifacts,
    cancer=CANCER,
    max_samples=MAX_SAMPLES,
    seed=SEED,
    use_score_col=SCORE_COL,
 )

print("X:", X.shape, X.dtype, "y:", y.shape, y.dtype)
print("Score range:", float(np.min(y)), "->", float(np.max(y)))
print("Cancer counts:", meta["cancer"].value_counts().sort_index().to_dict())
meta.head()

In [ ]:
# --- Train/val split + normalization (fit on train only) ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, random_state=SEED, shuffle=True
 )

mu = X_train.mean(axis=0, dtype=np.float64)
sigma = X_train.std(axis=0, dtype=np.float64) + EPS
X_train_n = ((X_train - mu) / sigma).astype(np.float32)
X_val_n = ((X_val - mu) / sigma).astype(np.float32)

norm_dir = run_dir / "norm"
norm_dir.mkdir(parents=True, exist_ok=True)
np.save(norm_dir / "mu.npy", mu.astype(np.float32))
np.save(norm_dir / "sigma.npy", sigma.astype(np.float32))

print("Train:", X_train_n.shape, "Val:", X_val_n.shape)
print("Saved norm + config into:", run_dir)

In [ ]:
# --- Training (metrics.csv + checkpoints + live W&B metrics) ---
model, best_ckpt, log_dir = fit_and_save_sae(
    X_train_n,
    sae_cfg=model_cfg,
    train_cfg=train_cfg,
    out_dir=run_dir,
    X_val=X_val_n,
    wandb_run=wandb_run,
)

print("Best checkpoint:", best_ckpt)
print("Log dir:", log_dir)

# Save final weights explicitly for the analysis notebook (use BEST when available)
if best_ckpt is not None and Path(best_ckpt).exists():
    ckpt = torch.load(best_ckpt, map_location="cpu")
    torch.save({"sae_cfg": ckpt["sae_cfg"], "state_dict": ckpt["state_dict"]}, run_dir / "final.pt")
else:
    torch.save({"sae_cfg": model_cfg.__dict__, "state_dict": model.state_dict()}, run_dir / "final.pt")
print("Saved:", run_dir / "final.pt")

metrics_df = load_metrics(run_dir / "metrics.csv")
print("Metrics rows:", len(metrics_df))

if wandb_run is not None:
    wandb_run.save(str(run_dir / "metrics.csv"), base_path=str(run_dir))
    model_artifact = wandb.Artifact(f"sae-{CANCER}-{run_name}", type="model")
    model_artifact.add_file(str(run_dir / "final.pt"), name="final.pt")
    if best_ckpt is not None and Path(best_ckpt).exists():
        model_artifact.add_file(str(best_ckpt), name="best.pt")
    wandb_run.log_artifact(model_artifact)

metrics_df.tail(5)


In [ ]:
metrics_df = load_metrics(run_dir / "metrics.csv")
plot_training_summary(metrics_df, title=f"SAE training: loss + latent metrics ({CANCER})", show_val=True)


In [ ]:
# --- Minimal plots + focused post-train diagnostics (reference-style) ---

# Load best weights (final.pt) and run diagnostics on a held-out batch
ckpt = torch.load(run_dir / "final.pt", map_location="cpu")
sae_cfg_loaded = SAEConfig(**ckpt["sae_cfg"])
model_diag = SparseAutoencoder(sae_cfg_loaded)
model_diag.load_state_dict(ckpt["state_dict"])
model_diag.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model_diag.to(device)

x0 = np.asarray(X_val_n, dtype=np.float32)
with torch.no_grad():
    xb = torch.from_numpy(x0).to(device)
    x_hat, z = model_diag.forward(xb)
    x1 = x_hat.detach().cpu().numpy().astype(np.float32)
    z_np = z.detach().cpu().numpy().astype(np.float32)

In [ ]:
# (2) Correlation structure shift: original vs reconstructed
sub_corr = int(x0.shape[0])
c0 = corr_offdiag(x0[:sub_corr])
c1 = corr_offdiag(x1[:sub_corr])
print("Sample–sample corr off-diag summary (orig):", summ(c0))
print("Sample–sample corr off-diag summary (recon):", summ(c1))
plt.figure(figsize=(6.6, 4.0))
plt.hist(c0, bins=60, alpha=0.55, label="orig", density=True)
plt.hist(c1, bins=60, alpha=0.55, label="recon", density=True)
plt.xlabel("pairwise correlation (off-diagonal)")
plt.ylabel("density")
plt.title("Correlation structure shift: original vs reconstructed")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# (3) Effective rank proxy (collapse/oversmoothing indicator)
sub_svd = int(x0.shape[0])
A0 = x0[:sub_svd] - x0[:sub_svd].mean(axis=1, keepdims=True)
A1 = x1[:sub_svd] - x1[:sub_svd].mean(axis=1, keepdims=True)
S0 = np.linalg.svd(A0, full_matrices=False, compute_uv=False)
S1 = np.linalg.svd(A1, full_matrices=False, compute_uv=False)
print("Effective rank (orig, recon):", effective_rank(S0), effective_rank(S1))
print("Top-1 variance fraction (orig, recon):", float((S0[0] ** 2) / (np.sum(S0**2) + 1e-12)), float((S1[0] ** 2) / (np.sum(S1**2) + 1e-12)))

In [ ]:
# --- Finish W&B run ---
if wandb_run is not None:
    wandb_run.finish()